In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '..')
import pandas as pd

from src.features import compute_hedge_ratio, compute_kalman_hedge, compute_calendar_features, compute_spread_vol

from src.labels import get_daily_vol, get_vertical_barrier, apply_pt_sl_on_t1, get_bins, get_avg_uniqueness


In [ ]:
# Import the full dataset and calculate the hedge ratio

df = pd.read_csv("../data/full_dataset.csv", index_col="date", parse_dates=True)
df = compute_hedge_ratio(df, 'close_corn', 'close_soybean')         # OLS
df = compute_kalman_hedge(df, 'close_corn', 'close_soybean')        # Kalman 3-state
df = compute_calendar_features(df)                                   # month, day_of_week
df = compute_spread_vol(df, 'close_corn', 'close_soybean', 'kf_hedge_ratio')  # EWMA vol

In [ ]:
# Following AFML book  we implement the labeling

# First getting the volatility of the spread
vol     = get_daily_vol(corn, soy, hedge_ratio)
t1      = get_vertical_barrier(df.index, num_days=150)
touches = apply_pt_sl_on_t1(corn, soy, hedge_ratio, t1, vol, pt_sl=[2, 2])
labels  = get_bins(touches, corn, soy, hedge_ratio)
weights = get_avg_uniqueness(labels, df.index)